[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/097.%20The%20Last-Mile%20Delivery%20%26%20Micro-Fulfillment%20Problem/P97-Tier-8_executed.ipynb)

---



# 97. The Last-Mile Delivery & Micro-Fulfillment Problem
## Tier 8: The Value-Aligned and Ethical Framework

### Purpose and scope

This notebook demonstrates a rule-based ethical assessment of a delivery plan. It first evaluates an efficiency-focused baseline against explicitly stated ethical thresholds. If the baseline violates any threshold, a rule-based re-optimization adjusts the relevant plan metrics, and the resulting plan is assessed again. The method is an illustrative policy framework, not a route-planning solver or a learned artificial-intelligence system.

The inputs are synthetic delivery requests, stakeholder descriptions, policy thresholds, and baseline plan metrics. The outputs are constraint assessments, an adjusted plan when needed, a comparison of selected metrics, and visual summaries. The example does not model road networks, actual route assignments, emissions inventories, worker schedules, privacy audits, stakeholder voting, or monetary operating costs. Its numerical metrics are illustrative and should not be interpreted as empirical findings.

### Modeling assumptions

- The plan-level metrics apply to every delivery assessed in the scenario.
- The ethical constitution contains six principles, while the seven operational constraints in this example measure four of them: environmental sustainability, social equity, worker welfare, and customer privacy. Transparency and fair competition are stated principles but are not operationalized as constraints here.
- A hard constraint is a non-negotiable threshold; a soft constraint expresses a preferred threshold. This demonstration checks both types and triggers re-optimization when either is violated.
- Re-optimization applies fixed, rule-based adjustments. It does not search a route space or prove that the resulting plan is globally optimal.
- Delivery locations are points in a synthetic coordinate plane. Euclidean distances are reported in coordinate units, not kilometers, and are not used as a substitute for the illustrative emissions ratio.

### Assessment and adjustment method

Let $C$ be the set of operational constraints and $P$ the set of ethical principles. For a plan $x$, each constraint $c \in C$ produces a measured value $m_c(x)$ and a compliance result $q_c(x)$. For a maximum-threshold constraint, compliance means $m_c(x) \leq \tau_c$; for a minimum-threshold constraint, it means $m_c(x) \geq \tau_c$, where $\tau_c$ is the stated threshold. A principle is compliant only when all constraints associated with that principle are compliant:

$$q_p(x) = \bigwedge_{c \in C_p} q_c(x), \qquad p \in P,\quad C_p = \{c \in C : c \text{ measures } p\}.$$

The baseline plan $x_0$ is assessed first. If any principle is non-compliant, the re-optimization rules produce an adjusted plan $x_1 = R(x_0, V(x_0))$, where $V(x_0)$ is the set of violated principles and $R$ applies the corresponding fixed adjustments. The final plan is assessed using the same thresholds. The implementation follows this assessment-then-adjustment sequence in `optimize_with_ethics`, `_identify_violations`, and `_ethical_reoptimization`.

The reported ethical score is a descriptive summary of principle-level compliance, with an additional penalty for violated principles that have hard constraints. It is not a universal measure of ethical quality. The efficiency change is the difference between the baseline and adjusted efficiency values; it is not a measured cost or a general efficiency–ethics optimum.

### What to examine

Check which constraints and principles fail for the baseline, whether the rule-based adjustment brings the measured values within their thresholds, and how the illustrative efficiency metric changes. A compliant result means only that the modeled metrics pass the encoded rules; it does not establish that a real delivery operation is fair, safe, sustainable, or privacy-preserving.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from enum import Enum
import random
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class EthicalPrinciple(Enum):
    ENVIRONMENTAL_SUSTAINABILITY = "environmental_sustainability"
    SOCIAL_EQUITY = "social_equity"
    WORKER_WELFARE = "worker_welfare"
    CUSTOMER_PRIVACY = "customer_privacy"
    FAIR_COMPETITION = "fair_competition"
    TRANSPARENCY = "transparency"


@dataclass
class EthicalConstraint:
    """A measurable policy threshold associated with an ethical principle."""

    principle: EthicalPrinciple
    constraint_type: str
    description: str
    metric_name: str
    threshold_value: float
    penalty_weight: float
    measurement_method: str
    comparison: str


@dataclass
class Stakeholder:
    """A group affected by delivery decisions and its stated interests."""

    id: int
    name: str
    type: str
    interests: List[str]
    utility_function: Dict[str, float]
    priority: float
    location: Optional[Tuple[float, float]] = None
    demographic_data: Optional[Dict] = None


@dataclass
class DeliveryDecision:
    """A synthetic delivery request used to describe the scenario."""

    id: int
    customer_id: int
    origin: Tuple[float, float]
    destination: Tuple[float, float]
    priority: int
    time_window: Tuple[datetime, datetime]
    package_size: float
    neighborhood_type: str
    environmental_sensitivity: float
    worker_safety_risk: float


@dataclass
class EthicalAssessment:
    """Constraint and stakeholder assessment for one delivery request."""

    decision_id: int
    constraint_compliance: Dict[EthicalPrinciple, bool]
    constraint_violations: List[EthicalPrinciple]
    constraint_measurements: Dict[str, float]
    stakeholder_impact: Dict[int, float]
    ethical_score: float
    trade_offs_identified: List[str]
    recommendation: str


class ConstitutionalAI:
    """Rule-based ethical assessment using a declared set of principles."""

    def __init__(self):
        self.constitution = self._establish_constitution()
        self.ethical_constraints = self._define_ethical_constraints()
        self.stakeholders: List[Stakeholder] = []

    def _establish_constitution(self) -> Dict[str, str]:
        """State the principles that guide the illustrative framework."""
        return {
            "principle_1": "Delivery decisions should minimize environmental impact.",
            "principle_2": "Service should be equitably distributed across socioeconomic areas.",
            "principle_3": "Worker safety and welfare should not be compromised for efficiency.",
            "principle_4": "Customer privacy and data protection are fundamental rights.",
            "principle_5": "Decision-making should be transparent and accountable.",
            "principle_6": "Fair competition and market integrity should be maintained.",
        }

    def _define_ethical_constraints(self) -> List[EthicalConstraint]:
        """Define the operational thresholds assessed in this example."""
        return [
            EthicalConstraint(
                EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY,
                "hard",
                "Emissions per delivery must not exceed 140% of the baseline.",
                "carbon_emissions",
                1.4,
                0.3,
                "Illustrative emissions ratio supplied with the plan.",
                "maximum",
            ),
            EthicalConstraint(
                EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY,
                "soft",
                "Prefer low-emission delivery methods when feasible.",
                "emission_preference",
                0.7,
                0.1,
                "Illustrative preference score supplied with the plan.",
                "minimum",
            ),
            EthicalConstraint(
                EthicalPrinciple.SOCIAL_EQUITY,
                "hard",
                "Service-quality differences between neighborhoods must not exceed 15%.",
                "service_equity",
                0.15,
                0.4,
                "Range of neighborhood service scores divided by the highest score.",
                "maximum",
            ),
            EthicalConstraint(
                EthicalPrinciple.SOCIAL_EQUITY,
                "soft",
                "Maintain a minimum service level in underserved areas.",
                "minimum_service",
                0.8,
                0.2,
                "Lowest modeled neighborhood service score.",
                "minimum",
            ),
            EthicalConstraint(
                EthicalPrinciple.WORKER_WELFARE,
                "hard",
                "Driving time must not exceed 10 hours per shift.",
                "worker_hours",
                10.0,
                0.5,
                "Illustrative hours per shift supplied with the plan.",
                "maximum",
            ),
            EthicalConstraint(
                EthicalPrinciple.WORKER_WELFARE,
                "soft",
                "Break scheduling should meet the stated compliance target.",
                "break_compliance",
                0.9,
                0.1,
                "Illustrative break-compliance score supplied with the plan.",
                "minimum",
            ),
            EthicalConstraint(
                EthicalPrinciple.CUSTOMER_PRIVACY,
                "hard",
                "The data-collection score must not exceed the minimization threshold.",
                "data_minimization",
                0.3,
                0.3,
                "Number of collected data fields divided by ten.",
                "maximum",
            ),
        ]

    def add_stakeholder(self, stakeholder: Stakeholder) -> None:
        """Register a stakeholder for the illustrative impact assessment."""
        self.stakeholders.append(stakeholder)

    def _check_constraint_compliance(
        self,
        proposed_solution: Dict,
        constraint: EthicalConstraint,
    ) -> Dict:
        """Measure one plan metric and compare it with its stated threshold."""
        if constraint.metric_name == "service_equity":
            service_levels = list(proposed_solution["neighborhood_service"].values())
            maximum = max(service_levels, default=0.0)
            measured_value = (
                (maximum - min(service_levels)) / maximum
                if maximum > 0 and service_levels
                else 0.0
            )
        elif constraint.metric_name == "minimum_service":
            service_levels = list(proposed_solution["neighborhood_service"].values())
            measured_value = min(service_levels, default=0.0)
        elif constraint.metric_name == "data_minimization":
            measured_value = len(proposed_solution["data_collected"]) / 10.0
        else:
            measured_value = float(proposed_solution[constraint.metric_name])

        if constraint.comparison == "maximum":
            complies = measured_value <= constraint.threshold_value
        else:
            complies = measured_value >= constraint.threshold_value

        return {
            "complies": complies,
            "measured_value": measured_value,
            "threshold": constraint.threshold_value,
        }

    def _calculate_stakeholder_impacts(
        self,
        decision: DeliveryDecision,
        proposed_solution: Dict,
        compliance: Dict[EthicalPrinciple, bool],
    ) -> Dict[int, float]:
        """Compute transparent, illustrative impact indicators by stakeholder."""
        impacts = {}
        for stakeholder in self.stakeholders:
            if stakeholder.type == "customer":
                impact = proposed_solution["service_quality"]
                if not compliance.get(EthicalPrinciple.CUSTOMER_PRIVACY, True):
                    impact -= 0.1
            elif stakeholder.type == "worker":
                impact = 1.0 - decision.worker_safety_risk
                if proposed_solution["worker_hours"] > 9.0:
                    impact -= 0.1
            elif stakeholder.type == "community":
                impact = proposed_solution["service_quality"]
                if not compliance.get(EthicalPrinciple.SOCIAL_EQUITY, True):
                    impact -= 0.1
            elif stakeholder.type == "environment":
                impact = 1.0 - proposed_solution["carbon_emissions"]
            elif stakeholder.type == "company":
                impact = proposed_solution["efficiency"]
                if not all(compliance.values()):
                    impact -= 0.1
            else:
                impact = 0.0
            impacts[stakeholder.id] = impact
        return impacts

    def _calculate_ethical_score(
        self,
        compliance: Dict[EthicalPrinciple, bool],
        violations: List[EthicalPrinciple],
    ) -> float:
        """Summarize measured principle compliance; this is not a universal ethics metric."""
        if not compliance:
            return 1.0
        base_score = sum(compliance.values()) / len(compliance)
        hard_principles = {
            constraint.principle
            for constraint in self.ethical_constraints
            if constraint.constraint_type == "hard"
        }
        hard_violations = len(set(violations) & hard_principles)
        penalty = 0.2 * hard_violations / max(len(hard_principles), 1)
        return max(0.0, base_score - penalty)

    def _generate_ethical_recommendation(
        self,
        violations: List[EthicalPrinciple],
        ethical_score: float,
    ) -> str:
        """Turn the rule-based assessment into a concise recommendation."""
        if not violations:
            return "All operationalized thresholds pass; review the unmeasured principles separately."
        if ethical_score >= 0.7:
            return "Thresholds are not all met; apply the relevant adjustments and reassess."
        return "Several operationalized thresholds fail; review the plan before implementation."

    def assess_decision_ethics(
        self,
        decision: DeliveryDecision,
        proposed_solution: Dict,
    ) -> EthicalAssessment:
        """Assess one decision against all operationalized plan-level constraints."""
        compliance = {}
        measurements = {}
        trade_offs = []

        for constraint in self.ethical_constraints:
            result = self._check_constraint_compliance(proposed_solution, constraint)
            measurements[constraint.metric_name] = result["measured_value"]
            previous = compliance.get(constraint.principle, True)
            compliance[constraint.principle] = previous and result["complies"]
            if not result["complies"]:
                trade_offs.append(
                    f"{constraint.description} Measured {result['measured_value']:.3f}; "
                    f"threshold {result['threshold']:.3f}."
                )

        violations = [principle for principle, passes in compliance.items() if not passes]
        score = self._calculate_ethical_score(compliance, violations)
        impacts = self._calculate_stakeholder_impacts(
            decision, proposed_solution, compliance
        )
        recommendation = self._generate_ethical_recommendation(violations, score)

        return EthicalAssessment(
            decision_id=decision.id,
            constraint_compliance=compliance,
            constraint_violations=violations,
            constraint_measurements=measurements,
            stakeholder_impact=impacts,
            ethical_score=score,
            trade_offs_identified=trade_offs,
            recommendation=recommendation,
        )


class EthicalOptimizer:
    """Assess a baseline plan and apply fixed adjustments for violated principles."""

    def __init__(self, constitutional_ai: ConstitutionalAI):
        self.ai = constitutional_ai
        self.optimization_history = []

    def optimize_with_ethics(self, decisions: List[DeliveryDecision]) -> Dict:
        """Run baseline assessment, rule-based adjustment, and final assessment."""
        if not decisions:
            raise ValueError("At least one delivery decision is required.")

        print("Running rule-based ethical assessment and adjustment...")
        original_solution = self._generate_baseline_solution(decisions)
        original_assessments = [
            self.ai.assess_decision_ethics(decision, original_solution)
            for decision in decisions
        ]
        violations = self._identify_violations(original_assessments)

        ethical_solution = None
        trade_off_analysis = None
        if violations:
            ethical_solution = self._ethical_reoptimization(decisions, violations)
            final_solution = ethical_solution
        else:
            final_solution = original_solution

        final_assessments = [
            self.ai.assess_decision_ethics(decision, final_solution)
            for decision in decisions
        ]
        if violations:
            trade_off_analysis = self._analyze_trade_offs(
                original_solution,
                final_solution,
                original_assessments,
                final_assessments,
            )

        self.optimization_history.append(
            {"baseline_violations": violations, "final_solution": final_solution}
        )
        return {
            "original_solution": original_solution,
            "original_assessments": original_assessments,
            "violations": violations,
            "ethical_optimization": ethical_solution,
            "trade_off_analysis": trade_off_analysis,
            "final_solution": final_solution,
            "final_assessments": final_assessments,
        }

    def _generate_baseline_solution(self, decisions: List[DeliveryDecision]) -> Dict:
        """Construct the illustrative efficiency-focused baseline plan."""
        neighborhood_quality = {
            "affluent": 0.95,
            "middle": 0.85,
            "low_income": 0.70,
        }
        neighborhood_counts = {name: 0 for name in neighborhood_quality}
        total_distance = 0.0
        for decision in decisions:
            if decision.neighborhood_type not in neighborhood_counts:
                raise ValueError(
                    f"Unknown neighborhood type: {decision.neighborhood_type}"
                )
            dx = decision.destination[0] - decision.origin[0]
            dy = decision.destination[1] - decision.origin[1]
            total_distance += float(np.hypot(dx, dy))
            neighborhood_counts[decision.neighborhood_type] += 1

        service = {
            neighborhood: neighborhood_quality[neighborhood]
            for neighborhood, count in neighborhood_counts.items()
            if count > 0
        }
        return {
            "total_distance": total_distance,
            "neighborhood_service": service,
            "service_quality": 0.8,
            "efficiency": 0.95,
            "carbon_emissions": 1.6,
            "emission_preference": 0.5,
            "worker_hours": 11.2,
            "break_compliance": 0.75,
            "data_collected": [
                "location",
                "time",
                "signature",
                "photo",
                "customer_id",
                "package_contents",
            ],
        }

    def _identify_violations(
        self,
        assessments: List[EthicalAssessment],
    ) -> List[EthicalPrinciple]:
        """Collect distinct violated principles across the assessed requests."""
        violations = {
            principle
            for assessment in assessments
            for principle in assessment.constraint_violations
        }
        return sorted(violations, key=lambda principle: principle.value)

    def _ethical_reoptimization(
        self,
        decisions: List[DeliveryDecision],
        violations: List[EthicalPrinciple],
    ) -> Dict:
        """Apply the example's fixed adjustment rule for each violated principle."""
        solution = self._generate_baseline_solution(decisions)

        if EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY in violations:
            solution["carbon_emissions"] = 1.2
            solution["emission_preference"] = 0.8
            solution["efficiency"] -= 0.05

        if EthicalPrinciple.SOCIAL_EQUITY in violations:
            solution["neighborhood_service"] = {
                "affluent": 0.85,
                "middle": 0.85,
                "low_income": 0.88,
            }
            solution["efficiency"] -= 0.07

        if EthicalPrinciple.WORKER_WELFARE in violations:
            solution["worker_hours"] = 9.1
            solution["break_compliance"] = 0.95
            solution["efficiency"] -= 0.03

        if EthicalPrinciple.CUSTOMER_PRIVACY in violations:
            solution["data_collected"] = ["location", "time"]
            solution["efficiency"] -= 0.02

        return solution

    def _analyze_trade_offs(
        self,
        original_solution: Dict,
        ethical_solution: Dict,
        original_assessments: List[EthicalAssessment],
        ethical_assessments: List[EthicalAssessment],
    ) -> Dict:
        """Compare modeled efficiency and compliance summaries."""
        efficiency_loss = (
            original_solution["efficiency"] - ethical_solution["efficiency"]
        )
        original_score = float(
            np.mean([assessment.ethical_score for assessment in original_assessments])
        )
        ethical_score = float(
            np.mean([assessment.ethical_score for assessment in ethical_assessments])
        )
        original_violating = sum(
            bool(assessment.constraint_violations)
            for assessment in original_assessments
        )
        final_violating = sum(
            bool(assessment.constraint_violations)
            for assessment in ethical_assessments
        )
        return {
            "efficiency_loss": efficiency_loss,
            "cost_proxy": efficiency_loss * 1000.0,
            "ethical_improvement": ethical_score - original_score,
            "violating_decisions_resolved": original_violating - final_violating,
        }


print("Ethical assessment framework is ready.")

In [ ]:
print("=" * 60)
print("INITIALIZING THE VALUE-ALIGNED FRAMEWORK")
print("=" * 60)

constitutional_ai = ConstitutionalAI()

stakeholders = [
    Stakeholder(
        id=1,
        name="Customer Association",
        type="customer",
        interests=["service_quality", "privacy", "fair_pricing", "reliability"],
        utility_function={
            "service_quality": 0.4,
            "privacy": 0.3,
            "fair_pricing": 0.2,
            "reliability": 0.1,
        },
        priority=0.9,
    ),
    Stakeholder(
        id=2,
        name="Delivery Workers Union",
        type="worker",
        interests=["safety", "fair_wages", "reasonable_hours", "job_security"],
        utility_function={
            "safety": 0.4,
            "fair_wages": 0.3,
            "reasonable_hours": 0.2,
            "job_security": 0.1,
        },
        priority=0.8,
    ),
    Stakeholder(
        id=3,
        name="Environmental Council",
        type="environment",
        interests=["carbon_reduction", "noise_pollution", "green_delivery"],
        utility_function={
            "carbon_reduction": 0.4,
            "noise_pollution": 0.2,
            "green_delivery": 0.4,
        },
        priority=0.7,
    ),
    Stakeholder(
        id=4,
        name="Community Alliance",
        type="community",
        interests=["service_equity", "local_jobs", "accessibility"],
        utility_function={
            "service_equity": 0.5,
            "local_jobs": 0.2,
            "accessibility": 0.3,
        },
        priority=0.8,
    ),
    Stakeholder(
        id=5,
        name="Company Management",
        type="company",
        interests=["profitability", "efficiency", "reputation"],
        utility_function={
            "profitability": 0.4,
            "efficiency": 0.4,
            "reputation": 0.2,
        },
        priority=0.9,
    ),
]

for stakeholder in stakeholders:
    constitutional_ai.add_stakeholder(stakeholder)

constraint_types = {
    constraint.constraint_type for constraint in constitutional_ai.ethical_constraints
}
print(f"Constitutional principles: {len(constitutional_ai.constitution)}")
print(f"Operational constraints: {len(constitutional_ai.ethical_constraints)}")
print(f"Represented stakeholder groups: {len(constitutional_ai.stakeholders)}")
print(f"Constraint categories: {', '.join(sorted(constraint_types))}")

print("\nCONSTITUTIONAL PRINCIPLES")
for index, principle in enumerate(constitutional_ai.constitution.values(), start=1):
    print(f"{index}. {principle}")

print("\nOPERATIONAL CONSTRAINTS")
for constraint in constitutional_ai.ethical_constraints:
    print(
        f"- {constraint.principle.value.replace('_', ' ').title()} "
        f"({constraint.constraint_type}): {constraint.description}"
    )

print("\nSTAKEHOLDERS")
for stakeholder in constitutional_ai.stakeholders:
    print(
        f"- {stakeholder.name} ({stakeholder.type}; "
        f"stated priority {stakeholder.priority:.1f})"
    )

In [ ]:
print("=" * 60)
print("CREATING A SYNTHETIC HOLIDAY DELIVERY SCENARIO")
print("=" * 60)

holiday_decisions = []
neighborhood_distribution = {
    "affluent": 0.30,
    "middle": 0.45,
    "low_income": 0.25,
}
total_deliveries = 100

for index in range(total_deliveries):
    draw = random.random()
    if draw < neighborhood_distribution["affluent"]:
        neighborhood = "affluent"
        destination = (random.uniform(60, 90), random.uniform(60, 90))
    elif draw < (
        neighborhood_distribution["affluent"] + neighborhood_distribution["middle"]
    ):
        neighborhood = "middle"
        destination = (random.uniform(30, 70), random.uniform(30, 70))
    else:
        neighborhood = "low_income"
        destination = (random.uniform(10, 40), random.uniform(10, 40))

    window_start = datetime.now() + timedelta(hours=random.uniform(0, 4))
    window_end = window_start + timedelta(hours=random.uniform(2, 6))
    holiday_decisions.append(
        DeliveryDecision(
            id=index + 1,
            customer_id=random.randint(1000, 9999),
            origin=(random.uniform(20, 80), random.uniform(20, 80)),
            destination=destination,
            priority=random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0],
            time_window=(window_start, window_end),
            package_size=random.uniform(0.1, 3.0),
            neighborhood_type=neighborhood,
            environmental_sensitivity=random.uniform(0.1, 0.9),
            worker_safety_risk=random.uniform(0.05, 0.4),
        )
    )

print(f"Synthetic delivery requests: {len(holiday_decisions)}")
for neighborhood in ("affluent", "middle", "low_income"):
    count = sum(
        decision.neighborhood_type == neighborhood
        for decision in holiday_decisions
    )
    print(f"{neighborhood.replace('_', ' ').title()} area: {count} requests")

print(
    "\nThe dilemma is represented by unequal baseline service-quality scores, "
    "not by a claim about traffic concentration. The framework will assess the "
    "declared plan metrics against the encoded thresholds."
)

ethical_optimizer = EthicalOptimizer(constitutional_ai)
print("\nRule-based ethical optimizer is ready.")

In [ ]:
optimization_results = ethical_optimizer.optimize_with_ethics(holiday_decisions)
original_solution = optimization_results["original_solution"]
final_solution = optimization_results["final_solution"]
original_assessments = optimization_results["original_assessments"]
final_assessments = optimization_results["final_assessments"]
trade_offs = optimization_results["trade_off_analysis"]

mean_baseline_score = float(
    np.mean([assessment.ethical_score for assessment in original_assessments])
)
mean_final_score = float(
    np.mean([assessment.ethical_score for assessment in final_assessments])
)
baseline_violations = optimization_results["violations"]
final_violations = ethical_optimizer._identify_violations(final_assessments)

print("\nBASELINE ASSESSMENT")
print(f"Mean descriptive ethical score: {mean_baseline_score:.3f}")
print(f"Violated principles: {len(baseline_violations)}")
for principle in baseline_violations:
    print(f"- {principle.value.replace('_', ' ').title()}")

if original_assessments:
    example_assessment = original_assessments[0]
    print("\nMeasured baseline values for one assessed request")
    for constraint in constitutional_ai.ethical_constraints:
        value = example_assessment.constraint_measurements[constraint.metric_name]
        status = example_assessment.constraint_compliance[constraint.principle]
        print(
            f"- {constraint.metric_name}: {value:.3f}; "
            f"threshold {constraint.comparison} {constraint.threshold_value:.3f}; "
            f"principle status: {'compliant' if status else 'non-compliant'}"
        )

print("\nFINAL PLAN ASSESSMENT")
print(f"Mean descriptive ethical score: {mean_final_score:.3f}")
print(
    "Remaining violated principles: "
    + (
        ", ".join(
            principle.value.replace("_", " ").title()
            for principle in final_violations
        )
        if final_violations
        else "None among the operationalized principles"
    )
)
print(f"Baseline efficiency metric: {original_solution['efficiency']:.3f}")
print(f"Final efficiency metric: {final_solution['efficiency']:.3f}")

if trade_offs is not None:
    print(f"Efficiency change: {trade_offs['efficiency_loss'] * 100:.1f} percentage points")
    print(f"Illustrative cost proxy: {trade_offs['cost_proxy']:.2f} units")
    print(
        "Change in descriptive ethical score: "
        f"{trade_offs['ethical_improvement'] * 100:.1f} percentage points"
    )
    print(
        "Requests changing from at least one violation to none: "
        f"{trade_offs['violating_decisions_resolved']}"
    )
else:
    print("The baseline passed the encoded thresholds; no adjustment was triggered.")

print(
    "\nInterpretation: these results describe the synthetic rule-based example. "
    "They are not measurements of real-world delivery performance or proof of "
    "overall ethical acceptability."
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(
    "Rule-Based Ethical Assessment of the Synthetic Delivery Plan",
    fontsize=15,
    fontweight="bold",
)

ax1, ax2, ax3, ax4 = axes.flat

principle_names = [
    principle.value.replace("_", " ").title()
    for principle in EthicalPrinciple
    if any(
        constraint.principle == principle
        for constraint in constitutional_ai.ethical_constraints
    )
]
baseline_rates = []
final_rates = []
for principle in EthicalPrinciple:
    if principle.value.replace("_", " ").title() not in principle_names:
        continue
    baseline_rates.append(
        np.mean(
            [assessment.constraint_compliance[principle] for assessment in original_assessments]
        )
    )
    final_rates.append(
        np.mean(
            [assessment.constraint_compliance[principle] for assessment in final_assessments]
        )
    )

positions = np.arange(len(principle_names))
bar_width = 0.38
ax1.bar(positions - bar_width / 2, baseline_rates, bar_width, label="Baseline")
ax1.bar(positions + bar_width / 2, final_rates, bar_width, label="Final plan")
ax1.set_xticks(positions)
ax1.set_xticklabels(principle_names, rotation=35, ha="right")
ax1.set_ylim(0, 1.05)
ax1.set_ylabel("Share of requests compliant")
ax1.set_title("Principle-Level Compliance")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

stakeholder_ids = [stakeholder.id for stakeholder in constitutional_ai.stakeholders]
stakeholder_names = [
    stakeholder.name.replace(" ", "\n")
    for stakeholder in constitutional_ai.stakeholders
]
baseline_impacts = [
    np.mean(
        [assessment.stakeholder_impact[stakeholder_id] for assessment in original_assessments]
    )
    for stakeholder_id in stakeholder_ids
]
final_impacts = [
    np.mean(
        [assessment.stakeholder_impact[stakeholder_id] for assessment in final_assessments]
    )
    for stakeholder_id in stakeholder_ids
]
positions = np.arange(len(stakeholder_names))
ax2.barh(positions - bar_width / 2, baseline_impacts, bar_width, label="Baseline")
ax2.barh(positions + bar_width / 2, final_impacts, bar_width, label="Final plan")
ax2.set_yticks(positions)
ax2.set_yticklabels(stakeholder_names)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_xlabel("Illustrative impact indicator")
ax2.set_title("Stakeholder Impact Indicators")
ax2.legend()
ax2.grid(axis="x", alpha=0.3)

metric_names = ["Efficiency", "Emissions ratio", "Service equity", "Worker hours"]
baseline_values = [
    original_solution["efficiency"],
    original_solution["carbon_emissions"],
    original_assessments[0].constraint_measurements["service_equity"],
    original_solution["worker_hours"],
]
final_values = [
    final_solution["efficiency"],
    final_solution["carbon_emissions"],
    final_assessments[0].constraint_measurements["service_equity"],
    final_solution["worker_hours"],
]
positions = np.arange(len(metric_names))
ax3.bar(positions - bar_width / 2, baseline_values, bar_width, label="Baseline")
ax3.bar(positions + bar_width / 2, final_values, bar_width, label="Final plan")
ax3.set_xticks(positions)
ax3.set_xticklabels(metric_names, rotation=20, ha="right")
ax3.set_ylabel("Metric value; units differ by metric")
ax3.set_title("Selected Plan Metrics")
ax3.legend()
ax3.grid(axis="y", alpha=0.3)

neighborhoods = list(final_solution["neighborhood_service"])
baseline_service = original_solution["neighborhood_service"]
final_service = final_solution["neighborhood_service"]
positions = np.arange(len(neighborhoods))
ax4.bar(
    positions - bar_width / 2,
    [baseline_service.get(name, 0.0) for name in neighborhoods],
    bar_width,
    label="Baseline",
)
ax4.bar(
    positions + bar_width / 2,
    [final_service.get(name, 0.0) for name in neighborhoods],
    bar_width,
    label="Final plan",
)
ax4.axhline(0.8, color="black", linestyle="--", label="Minimum service target")
ax4.set_xticks(positions)
ax4.set_xticklabels([name.replace("_", " ").title() for name in neighborhoods])
ax4.set_ylim(0, 1.05)
ax4.set_ylabel("Illustrative service-quality score")
ax4.set_title("Neighborhood Service Scores")
ax4.legend()
ax4.grid(axis="y", alpha=0.3)

fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

In [ ]:
print("=" * 60)
print("FRAMEWORK SUMMARY")
print("=" * 60)

print("\nOperationalized principle compliance")
for principle in EthicalPrinciple:
    if not any(
        constraint.principle == principle
        for constraint in constitutional_ai.ethical_constraints
    ):
        continue
    rate = np.mean(
        [assessment.constraint_compliance[principle] for assessment in final_assessments]
    )
    print(f"- {principle.value.replace('_', ' ').title()}: {rate:.1%}")

print("\nMean descriptive ethical score")
print(
    f"- Baseline: {mean_baseline_score:.3f}; "
    f"final plan: {mean_final_score:.3f}"
)

print("\nStakeholder impact indicators for the final plan")
for stakeholder in constitutional_ai.stakeholders:
    mean_impact = np.mean(
        [
            assessment.stakeholder_impact[stakeholder.id]
            for assessment in final_assessments
        ]
    )
    print(f"- {stakeholder.name}: {mean_impact:+.3f}")

hard_constraint_count = sum(
    constraint.constraint_type == "hard"
    for constraint in constitutional_ai.ethical_constraints
)
soft_constraint_count = sum(
    constraint.constraint_type == "soft"
    for constraint in constitutional_ai.ethical_constraints
)
unmeasured_principles = [
    principle.value.replace("_", " ").title()
    for principle in EthicalPrinciple
    if not any(
        constraint.principle == principle
        for constraint in constitutional_ai.ethical_constraints
    )
]

print("\nInterpretation and limitations")
print(f"- Encoded constraint types: {hard_constraint_count} hard and {soft_constraint_count} soft.")
print(
    "- Principles without operational constraints: "
    + (", ".join(unmeasured_principles) if unmeasured_principles else "None")
    + "."
)
print(
    "- Stakeholder indicators are simple illustrative rules; stakeholder priorities "
    "and utility weights do not enter a formal multi-objective optimization."
)
print(
    "- The adjustment is a fixed policy rule, not a search for a globally optimal "
    "delivery plan. Its thresholds and metrics require domain validation before use."
)

## Role, strengths, and limitations of this tier

This tier adds explicit ethical principles and measurable policy thresholds to the delivery-planning discussion. It treats selected requirements as hard constraints and others as soft targets, then uses a fixed adjustment rule when the assessed plan fails a threshold. This makes some trade-offs visible, but it does not constitute a complete ethical decision procedure or a general multi-objective optimizer.

### Strengths

- Makes selected environmental, equity, worker-welfare, and privacy requirements explicit.
- Reports which encoded thresholds fail and how the illustrative plan metrics change.
- Represents several affected stakeholder groups and reports simple impact indicators.
- Separates stated constitutional principles from the subset that has an operational measurement in this example.

### Limitations

- The synthetic scores, thresholds, and adjustment amounts are illustrative and require evidence-based calibration.
- Stakeholder priorities and utility weights are recorded but are not combined into an optimization objective.
- The rule-based adjustment does not assign actual vehicles, construct routes, or guarantee feasibility for operational schedules.
- The ethical score is a summary of encoded compliance, not a validated measure of moral or social value.
- Transparency and fair competition are stated principles but are not tested by operational constraints here.
- Real assessments would require carefully governed data, stakeholder participation, validated metrics, and ongoing review.

### Appropriate use

Use this framework as a teaching example of how explicit ethical thresholds can be assessed alongside an efficiency-focused plan. For practical applications, first define measurable and auditable indicators with affected stakeholders, verify the data and thresholds, and ensure that any plan adjustment respects operational and legal requirements.